In [ ]:
# ============================================================
# Exercise 2: Demosaicing & HDR
# ============================================================

# If needed, run this once:
# %pip install rawpy opencv-python matplotlib pillow scipy

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import rawpy


# ============================================================
# General settings
# ============================================================

BASE_DIR = Path.cwd()  # Start Jupyter from this group directory.
DATA_ROOT = BASE_DIR / "exercise_2_data"
OUT_DIR = BASE_DIR / "outputs_ex02"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-8

# Exercise 1 result:
# Bayer pattern found from IMG_9939.npy:
# G B
# R G
BAYER_PATTERN = "GBRG"


def find_file(filename):
    """
    Find a file inside the exercise_2_data folder.
    """
    results = list(DATA_ROOT.rglob(filename))
    if not results:
        results = list(BASE_DIR.rglob(filename))
    if not results:
        raise FileNotFoundError(f"Could not find {filename}")
    return results[0]


def read_raw_visible(raw_path):
    """
    Read CR3 raw sensor data as visible Bayer mosaic.
    Output is float32.
    """
    raw = rawpy.imread(str(raw_path))
    arr = np.array(raw.raw_image_visible, dtype=np.float32)
    raw.close()
    return arr


def unit_to_uint8(img):
    """
    Convert [0,1] image to uint8 only for saving.
    """
    return np.round(np.clip(img, 0, 1) * 255).astype(np.uint8)


def save_unit_image(img_unit, out_path, quality=99):
    """
    Save [0,1] RGB image as JPG.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(unit_to_uint8(img_unit)).save(out_path, quality=quality, subsampling=0)
    print("Saved:", out_path)


def normalize_percentile(data, p_low=0.01, p_high=99.99):
    """
    Normalize using percentiles, as requested in the exercise sheet.
    """
    a = np.percentile(data, p_low)
    b = np.percentile(data, p_high)

    out = (data - a) / (b - a + EPS)
    out = np.clip(out, 0, 1)

    return out.astype(np.float32)


def make_bayer_masks(shape, pattern="GBRG"):
    """
    Create R, G, B masks for the Bayer mosaic.
    For GBRG:
        G B
        R G
    """
    h, w = shape

    tile = np.array([
        [pattern[0], pattern[1]],
        [pattern[2], pattern[3]]
    ])

    rows, cols = np.indices((h, w))
    grid = tile[rows % 2, cols % 2]

    mask_r = (grid == "R").astype(np.float32)
    mask_g = (grid == "G").astype(np.float32)
    mask_b = (grid == "B").astype(np.float32)

    return mask_r, mask_g, mask_b


def demosaic_simple(mosaic, pattern="GBRG", kernel_size=3):
    """
    Exercise 2 demosaicing method from lecture:

        C = ((M_c * X) convolved with K) / (M_c convolved with K)

    X   = raw Bayer mosaic
    M_c = color mask
    K   = averaging kernel

    Output is float32 RGB.
    """
    mosaic = mosaic.astype(np.float32)

    mask_r, mask_g, mask_b = make_bayer_masks(mosaic.shape, pattern)
    kernel = np.ones((kernel_size, kernel_size), dtype=np.float32)

    channels = []

    for mask in [mask_r, mask_g, mask_b]:
        numerator = cv2.filter2D(
            mosaic * mask,
            ddepth=cv2.CV_32F,
            kernel=kernel,
            borderType=cv2.BORDER_REFLECT
        )

        denominator = cv2.filter2D(
            mask,
            ddepth=cv2.CV_32F,
            kernel=kernel,
            borderType=cv2.BORDER_REFLECT
        )

        channel = numerator / (denominator + EPS)
        channels.append(channel)

    rgb = np.stack(channels, axis=-1).astype(np.float32)
    return rgb


def apply_gamma(rgb, gamma=0.3):
    """
    Exercise 3 gamma correction.
    First normalize with percentiles, then apply y = x^gamma.
    """
    unit = normalize_percentile(rgb, 0.01, 99.99)
    corrected = unit ** gamma
    corrected = np.clip(corrected, 0, 1)
    return corrected.astype(np.float32)


def apply_log_curve(rgb, strength=8.0):
    """
    Exercise 3 second curve, different from gamma.
    """
    unit = normalize_percentile(rgb, 0.01, 99.99)
    corrected = np.log1p(strength * unit) / np.log1p(strength)
    corrected = np.clip(corrected, 0, 1)
    return corrected.astype(np.float32)


def gray_world(rgb):
    """
    Exercise 4 gray-world white balance.

    Assumption:
    average color should be gray, so R/G/B means should become similar.
    """
    rgb = rgb.astype(np.float32)

    channel_means = rgb.reshape(-1, 3).mean(axis=0)
    image_mean = channel_means.mean()

    gains = image_mean / (channel_means + EPS)
    balanced = rgb * gains.reshape(1, 1, 3)

    return balanced.astype(np.float32), gains


def minmax_to_unit(data):
    """
    Normalize data to [0,1] using min/max.
    Used after log HDR compression.
    """
    dmin = data.min()
    dmax = data.max()
    out = (data - dmin) / (dmax - dmin + EPS)
    return np.clip(out, 0, 1).astype(np.float32)

In [ ]:
# ============================================================
# Exercise 1: Investigate Bayer Pattern
# ============================================================

raw_9939 = np.load(find_file("IMG_9939.npy")).astype(np.float32)

print("IMG_9939.npy")
print("shape:", raw_9939.shape)
print("dtype:", raw_9939.dtype)
print("min:", raw_9939.min())
print("max:", raw_9939.max())
print("mean:", raw_9939.mean())

print("\nMean values for each 2x2 Bayer position:")

for r in [0, 1]:
    for c in [0, 1]:
        part = raw_9939[r::2, c::2]
        print(f"position ({r},{c}) mean = {part.mean():.2f}")

print("\nManual conclusion:")
print("Bayer pattern = GBRG")
print("2x2 tile:")
print("G B")
print("R G")

# Visual check
rgb_9939 = demosaic_simple(raw_9939, pattern=BAYER_PATTERN)
rgb_9939_preview = apply_gamma(rgb_9939, gamma=0.45)

save_unit_image(
    rgb_9939_preview,
    OUT_DIR / "ex01_IMG_9939_bayer_GBRG_preview.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(rgb_9939_preview)
plt.title("Exercise 1: IMG_9939 demosaiced with GBRG")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Exercise 2: Implement Demosaicing Algorithm
# ============================================================

raw_4782 = read_raw_visible(find_file("IMG_4782.CR3"))

print("IMG_4782.CR3")
print("shape:", raw_4782.shape)
print("dtype:", raw_4782.dtype)
print("min:", raw_4782.min())
print("max:", raw_4782.max())

rgb_4782 = demosaic_simple(
    raw_4782,
    pattern=BAYER_PATTERN,
    kernel_size=3
)

rgb_4782_preview = normalize_percentile(rgb_4782, 0.01, 99.99)

save_unit_image(
    rgb_4782_preview,
    OUT_DIR / "ex02_IMG_4782_demosaiced_only.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(rgb_4782_preview)
plt.title("Exercise 2: Demosaiced only")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Exercise 3: Improve Luminosity
# Gamma correction + one other curve
# ============================================================

# Gamma correction with gamma = 0.3
gamma_4782 = apply_gamma(
    rgb_4782,
    gamma=0.3
)

save_unit_image(
    gamma_4782,
    OUT_DIR / "ex03_IMG_4782_gamma_0_3.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(gamma_4782)
plt.title("Exercise 3: Gamma correction, gamma = 0.3")
plt.axis("off")
plt.show()


# Another curve: logarithmic curve
log_curve_4782 = apply_log_curve(
    rgb_4782,
    strength=8.0
)

save_unit_image(
    log_curve_4782,
    OUT_DIR / "ex03_IMG_4782_log_curve.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(log_curve_4782)
plt.title("Exercise 3: Logarithmic luminosity curve")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Exercise 4: White Balance with Gray World
# ============================================================

wb_4782, wb_gains = gray_world(rgb_4782)

print("Gray-world gains [R, G, B]:", wb_gains)

wb_gamma_4782 = apply_gamma(
    wb_4782,
    gamma=0.3
)

save_unit_image(
    wb_gamma_4782,
    OUT_DIR / "ex04_IMG_4782_gray_world_gamma.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(wb_gamma_4782)
plt.title("Exercise 4: Gray-world white balance + gamma")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Exercise 5: Show Sensor Data is Linear
# ============================================================

linear_files = [
    "IMG_3044.CR3",
    "IMG_3045.CR3",
    "IMG_3046.CR3",
    "IMG_3047.CR3",
    "IMG_3048.CR3",
    "IMG_3049.CR3"
]

exposure_times = np.array(
    [1/10, 1/20, 1/40, 1/80, 1/160, 1/320],
    dtype=np.float32
)

raw_means = []

for filename in linear_files:
    raw_img = read_raw_visible(find_file(filename))
    mean_value = raw_img.mean()
    raw_means.append(mean_value)
    print(filename, "mean raw value:", mean_value)

raw_means = np.array(raw_means, dtype=np.float32)

slope, intercept = np.polyfit(exposure_times, raw_means, deg=1)
fit_values = slope * exposure_times + intercept

ss_res = np.sum((raw_means - fit_values) ** 2)
ss_tot = np.sum((raw_means - raw_means.mean()) ** 2)
r2 = 1 - ss_res / (ss_tot + EPS)

print("slope:", slope)
print("intercept:", intercept)
print("R²:", r2)

plt.figure(figsize=(7, 5))
plt.scatter(exposure_times, raw_means, label="raw mean")
plt.plot(exposure_times, fit_values, label=f"linear fit, R²={r2:.5f}")
plt.xlabel("Exposure time [seconds]")
plt.ylabel("Average raw sensor value")
plt.title("Exercise 5: Sensor data is linear")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "ex05_sensor_linearity_plot.png", dpi=200)
plt.show()

In [ ]:
# ============================================================
# Exercise 6: Initial HDR Implementation
# ============================================================

def combine_hdr_raw(raw_paths, threshold_fraction=0.8):
    """
    HDR combination method from the lecture.

    00.CR3 is the longest exposure.
    Every next image has half the exposure time.
    Therefore:
        00.CR3 scale = 1
        01.CR3 scale = 2
        02.CR3 scale = 4
        ...
    """
    raw_paths = [Path(p) for p in raw_paths]

    h = read_raw_visible(raw_paths[0]).astype(np.float32)
    base_max = h.max()

    print("Base image:", raw_paths[0].name)
    print("Base max:", base_max)

    for k, path in enumerate(raw_paths[1:], start=1):
        i = read_raw_visible(path).astype(np.float32)

        scale = 2 ** k
        i_scaled = i * scale

        threshold = threshold_fraction * base_max * (2 ** (k - 1))

        mask = h > threshold

        print(
            path.name,
            "| scale:", scale,
            "| threshold:", threshold,
            "| replaced:", f"{mask.mean() * 100:.2f}%"
        )

        h[mask] = i_scaled[mask]

    return h


hdr_paths = [
    find_file(f"{i:02d}.CR3")
    for i in range(11)
]

hdr_raw = combine_hdr_raw(
    hdr_paths,
    threshold_fraction=0.8
)

hdr_rgb = demosaic_simple(
    hdr_raw,
    pattern=BAYER_PATTERN,
    kernel_size=3
)

hdr_wb_rgb, hdr_gains = gray_world(hdr_rgb)

print("HDR gray-world gains [R, G, B]:", hdr_gains)

hdr_log = np.log1p(np.maximum(hdr_wb_rgb, 0))
hdr_log_unit = minmax_to_unit(hdr_log)

save_unit_image(
    hdr_log_unit,
    OUT_DIR / "ex06_HDR_log_tonemap.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(hdr_log_unit)
plt.title("Exercise 6: HDR log tone mapping")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Exercise 7: iCAM06
# ============================================================

def icam06_tonemap(
    rgb_hdr,
    output_range=4.0,
    d=5,
    sigma_color=0.4,
    sigma_space=16.0,
    detail_strength=1.0
):
    """
    iCAM06 tone mapping based on the lecture slides.
    """
    rgb = np.maximum(rgb_hdr.astype(np.float32), EPS)

    R = rgb[:, :, 0]
    G = rgb[:, :, 1]
    B = rgb[:, :, 2]

    input_intensity = (20 * R + 40 * G + B) / 61.0
    input_intensity = np.maximum(input_intensity, EPS)

    chroma = rgb / input_intensity[:, :, None]

    log_intensity = np.log(input_intensity).astype(np.float32)

    log_base = cv2.bilateralFilter(
        log_intensity,
        d=d,
        sigmaColor=sigma_color,
        sigmaSpace=sigma_space,
        borderType=cv2.BORDER_REFLECT
    )

    log_detail = log_intensity - log_base

    compression = np.log(output_range) / (log_base.max() - log_base.min() + EPS)
    log_offset = -log_base.max() * compression

    output_intensity = np.exp(
        log_base * compression
        + log_offset
        + detail_strength * log_detail
    )

    out_rgb = chroma * output_intensity[:, :, None]

    return out_rgb.astype(np.float32)


icam_1 = icam06_tonemap(
    hdr_wb_rgb,
    output_range=4.0,
    d=5,
    sigma_color=0.4,
    sigma_space=16.0,
    detail_strength=1.0
)

icam_1_unit = normalize_percentile(icam_1, 0.01, 99.99)

save_unit_image(
    icam_1_unit,
    OUT_DIR / "ex07_iCAM06_setting_1.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(icam_1_unit)
plt.title("Exercise 7: iCAM06 setting 1")
plt.axis("off")
plt.show()


icam_2 = icam06_tonemap(
    hdr_wb_rgb,
    output_range=6.0,
    d=7,
    sigma_color=0.6,
    sigma_space=24.0,
    detail_strength=1.2
)

icam_2_unit = normalize_percentile(icam_2, 0.01, 99.99)

save_unit_image(
    icam_2_unit,
    OUT_DIR / "ex07_iCAM06_setting_2.jpg"
)

plt.figure(figsize=(8, 6))
plt.imshow(icam_2_unit)
plt.title("Exercise 7: iCAM06 setting 2")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Exercise 8: Final function process_raw
# ============================================================

def process_raw(raw_path, output_jpg_path):
    """
    Required function for Exercise 8.

    Input:
        raw_path: path to CR3 raw file

    Output:
        output_jpg_path: path where JPG image will be saved

    Pipeline:
        CR3 raw
        -> visible Bayer mosaic
        -> demosaicing
        -> gray-world white balance
        -> gamma correction
        -> slight export enhancement
        -> high quality JPG
    """
    raw_path = Path(raw_path)
    output_jpg_path = Path(output_jpg_path)

    output_jpg_path.parent.mkdir(parents=True, exist_ok=True)

    raw_mosaic = read_raw_visible(raw_path)

    rgb = demosaic_simple(
        raw_mosaic,
        pattern=BAYER_PATTERN,
        kernel_size=3
    )

    rgb_wb, gains = gray_world(rgb)

    print("process_raw gray-world gains [R, G, B]:", gains)

    img_unit = apply_gamma(
        rgb_wb,
        gamma=0.32
    )

    img8 = unit_to_uint8(img_unit)

    # Slight saturation correction for final JPG quality.
    hsv = cv2.cvtColor(img8, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.12, 0, 255)
    img8 = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # Mild sharpening.
    blur = cv2.GaussianBlur(img8, ksize=(0, 0), sigmaX=1.0)
    img8 = cv2.addWeighted(img8, 1.08, blur, -0.08, 0)

    Image.fromarray(img8).save(
        output_jpg_path,
        quality=99,
        subsampling=0
    )

    print("Saved:", output_jpg_path)


# Test Exercise 8 on IMG_4782.CR3
process_raw(
    find_file("IMG_4782.CR3"),
    OUT_DIR / "ex08_process_raw_IMG_4782.jpg"
)